# 🛠️ Construct Dataset Manually (WRDS + GXZ Predictors)
Use this section if you have access to WRDS and want to reconstruct the dataset from scratch using CRSP + predictors from Dacheng Xiu’s website.

In [ ]:
# --- Imports & Setup ---
import wrds
import pandas as pd
import os
import numpy as np

# --- Connect to WRDS ---
wrds_db = wrds.Connection()

# --- Download CRSP monthly returns ---
crsp = wrds_db.raw_sql("""
    SELECT permno, date, ret, shrout, prc, altprc, vol, retx
    FROM crsp.msf
    WHERE date >= '1957-01-01'
""")

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['yyyymm'] = crsp['date'].dt.year * 100 + crsp['date'].dt.month
crsp['me'] = crsp['prc'].abs() * crsp['shrout']
crsp['logme'] = np.log(crsp['me'])

# --- Load GXZ predictors from data_share.csv ---
chars = pd.read_csv("path/to/data_share.csv")  # 🔧 Update this path
chars.columns = chars.columns.str.lower()
chars['date'] = pd.to_datetime(chars['date'], format='%Y%m%d')
chars['yyyymm'] = chars['date'].dt.year * 100 + chars['date'].dt.month

# --- Merge CRSP with predictors ---
merged = pd.merge(chars, crsp, on=['permno', 'yyyymm'], how='inner')
merged.columns = merged.columns.str.lower()
merged = merged.fillna(0)

# --- Standardize features cross-sectionally by month ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in merged.columns if col not in exclude_cols]
merged[feature_cols] = merged.groupby('yyyymm')[feature_cols].transform(
    lambda x: (x - x.mean()) / x.std()
)
merged = merged.fillna(0)

# --- Save yearly .parquet files ---
os.makedirs("dataset_yearly_parquet", exist_ok=True)
merged['year'] = merged['yyyymm'] // 100

for year in sorted(merged['year'].unique()):
    chunk = merged[merged['year'] == year]
    filename = f"dataset_yearly_parquet/prepared_{year}.parquet"
    chunk.to_parquet(filename, index=False)
    print(f"Saved {filename} with shape {chunk.shape}")

# 💾 Load Full Dataset from Prebuilt Parquet Files and Run Diagnostics

In [ ]:
# --- Imports ---
import pandas as pd
import glob
from sklearn.linear_model import LinearRegression
import numpy as np
import os

# --- Load and combine all yearly parquet files ---
parquet_path = "INSERT PATHNAME FOR FOLDER HOLDING PARQUETS"
all_files = glob.glob(os.path.join(parquet_path, "*.parquet"))

df = pd.concat([pd.read_parquet(f) for f in all_files], ignore_index=True)

print(f"✅ Full dataset shape: {df.shape}")
print(f"Years covered: {df['yyyymm'].min() // 100} to {df['yyyymm'].max() // 100}")

# --- Identify predictors ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
predictor_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of predictors: {len(predictor_cols)}")

# --- Diagnostic 1: Check for duplicate permno + yyyymm combinations ---
dupes = df.duplicated(subset=['permno', 'yyyymm'])
print(f"Duplicate rows: {dupes.sum()}")

# --- Diagnostic 2: Check for missing returns ---
missing_returns = df['ret'].isna().sum()
print(f"Missing return values: {missing_returns}")

# --- Diagnostic 3: Standardization check for a few predictors ---
sample_cols = predictor_cols[:5]
standardization_stats = df.groupby('yyyymm')[sample_cols].agg(['mean', 'std'])
print("\nStandardization check (first few months):")
print(standardization_stats.head())

# --- Diagnostic 4: Check that key GXZ factors are present ---
required = ['logme', 'bm', 'mom12m']
for var in required:
    assert var in df.columns, f"❌ Missing required factor: {var}"
print("✅ All key GXZ benchmark factors are present.")

# --- Optional: Run benchmark 3-factor OLS to verify alignment with GXZ ---
X = df[required]
y = df['ret']
mask = X.notnull().all(axis=1) & y.notnull()
model = LinearRegression()
model.fit(X[mask], y[mask])
r2 = model.score(X[mask], y[mask])
print(f"📊 Benchmark 3-factor OLS R²: {r2:.4%}")